# Memory Spilling: Diagnosing Spill (Memory) and Spill (Disk)

**Module**: 06_memory_and_storage  
**Notebook**: 05_spill_to_disk.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: What causes memory spill during shuffles and sorts, how to diagnose spill in Spark UI, and how to eliminate it.
- **Why does this topic exist?**: Spilling destroys query performance due to disk serialization and I/O bottlenecks.
- **What problem does it solve?**: Queries slowing down 10x due to gigabytes of shuffle spill.
- **Where is it used in real-world Data Engineering?**: Fixing production stragglers in heavy aggregations.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Memory architecture, shuffle internals |
| **Required depth** | Working knowledge: can read spill metrics and reduce spill |
| **Data Engineer** | Diagnose spill and reduce it through projection, partitioning, and memory tuning. |
| **Senior Data Engineer** | Distinguish unavoidable spill from a design problem. |
| **Spark specialist** | Reason about execution memory, sort buffers, and disk I/O amplification. |
| **Next topic** | `06_memory_and_storage/06_broadcast_variables.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spill is Spark's safety valve: when execution memory cannot hold a sort or aggregation buffer, it writes sorted
runs to disk and merges them later. Some spill is normal on large shuffles. Persistent, heavy spill is a signal that
the job is moving too much data or using partitions that are too large for the memory budget.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Spill (memory)** | Serialized bytes before disk write | The in-memory size of the spilled buffer |
| **Spill (disk)** | Bytes written to disk | Can differ from memory size due to serialization/compression |
| **Execution memory** | Operator working memory | Sort buffers, hash maps, aggregation state |
| **Spill file** | Temporary disk file | Written under the local Spark directory |
| **Disk I/O amplification** | Extra read/write cost | Multiple spills and merges increase it |
| **Partition size** | Input to one task | Larger partitions need more execution memory |

### How it works, step by step

1. An operator requests execution memory for its buffer.
2. If the request cannot be satisfied, Spark spills existing data to disk.
3. The spilled run is sorted/serialized and written to a local file.
4. The operator continues with a fresh buffer.
5. At the end, spilled runs are merged to produce the final result.
6. Heavy spill increases disk I/O and can dominate stage time.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Spill (Memory) vs Spill (Disk)

- **Spill (Memory)**: Size of data in RAM before flushing.
- **Spill (Disk)**: Compressed size written to worker scratch disk.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spill_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("Check Stages tab in Spark UI for Spill (Memory) and Spill (Disk) columns.")
orders.groupBy("customer_id").count().show(3)

---

## Experiment: Eliminating Spill by Increasing Partitions

- **1. What are we testing?**: Show that increasing partition count reduces partition size below memory spill threshold.
- **2. What do we expect?**: Spill disappears when partitions are smaller.

In [ ]:
print('Increasing shuffle partitions from 2 to 16 reduces memory required per task by 8x.')

**3. What happened?**: Smaller partitions fit in execution memory.  
**4. Why did it happen?**: Each task processes fewer records.  
**5. Practical Engineering Lesson**: If a stage spills to disk, increase partition count or executor memory.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Sort-merge join with heavy spill

- **Industry**: Logistics
- **Scenario**: A nightly shipment join spills hundreds of gigabytes on each reduce task.
- **Problem**: Each reduce partition is too large for the execution memory budget, often because the partition count is too low or a few keys are hot.
- **Why this concept is relevant**: Increasing shuffle partitions and reducing row width can bring each partition within budget.
- **Implementation idea**: Measure spill per task, shuffle read, and partition-size distribution before and after.
- **Expected result**: A stage with bounded spill and shorter duration.
- **Alternative**: Increase executor memory.
- **Trade-offs**: More memory can absorb the spill but raises cost and may not address skew.

### Use Case 2: Aggregation with wide rows

- **Industry**: Healthcare
- **Scenario**: A claims aggregation spills despite a modest row count.
- **Problem**: Each row carries many unused string columns, so the in-memory sort buffer fills quickly.
- **Why this concept is relevant**: Projecting only the necessary columns before the aggregation reduces the bytes per row and the memory footprint.
- **Implementation idea**: Compare spill and shuffle write before and after projection.
- **Expected result**: A large reduction in spill with no change in the result.
- **Alternative**: Compression and a larger buffer.
- **Trade-offs**: Compression reduces disk bytes but still requires memory for the working set.

---

## Business Value

- Lower disk I/O and shorter stages.
- Better executor utilization.
- More predictable runtime for large jobs.
- Clear evidence for capacity planning.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Some spill is normal; investigate persistent or growing spill.
- Project columns and filter rows before wide operations.
- Check partition sizes and skew, not only total data size.
- Monitor disk throughput and local disk capacity.
- Do not assume more memory is always the right fix.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Reduce spill by projecting columns

**Steps**

1. Run an aggregation over a wide DataFrame and record spill metrics.
2. Select only the columns needed for the aggregation and repeat.
3. Compare spill, shuffle write, and stage duration.

**What to observe**: Less data per row reduces the memory required by the aggregation buffer.

**Healthy result**: Spill falls substantially and runtime improves.

**Problematic result**: Spill remains high because the partition count is too low.

**Engineering takeaway**: Reduce bytes before increasing memory.

---

## Trade-offs and Alternatives

### Advantages

- Prevents OOM
- Enables larger-than-memory operations
- Automatic

### Limitations

- Adds disk I/O
- Can dominate runtime
- Temporary files consume local disk

### When to use it

- Large sorts
- Large aggregations
- Skewed joins

### When not to use it

- Ignoring sustained spill
- Spilling because of wide rows
- Using spill as a sizing strategy

### Alternatives inside Spark

- Execution memory
- Sort-based shuffle
- AQE

### Alternatives outside Spark

- Larger instances
- Warehouse engines
- Pre-aggregation

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Find Spill Metrics in UI
**Problem Statement**: Locate Spill columns in UI.  
**Input / Context**: UI

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Calculate Spill Compression Ratio
**Problem Statement**: Explain why Spill (Disk) is smaller than Spill (Memory).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tuning Shuffle Buffer
**Problem Statement**: Config for shuffle memory fraction.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Spill that appears only at peak hours

**Scenario**: A job spills heavily during peak ingestion but not during off-peak runs.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Compare input partition sizes at peak and off-peak.
2. Check whether a few keys become hot during peak.
3. Look at local disk throughput and free space.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Find Spill Metrics in UI):
```python
print('Under Stage metrics table in localhost:4040/stages')
```

#### Solution 2 (Calculate Spill Compression Ratio):
```python
print('Spill (Disk) is compressed with LZ4/Snappy codec.')
```

#### Solution 3 (Tuning Shuffle Buffer):
```python
print('spark.memory.fraction')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Spill Impact**: Causes severe disk I/O latency.
- **Fix**: Increase partitions or increase executor memory.

### Essential Technical Terms
- `Spill`: Flushing in-memory buffers to disk.
- `AppendOnlyMap`: In-memory shuffle buffer.

### Common Engineering Mistakes
- **Ignoring non-zero spill**: Indicates execution memory pressure.

### Interview Questions on This Topic
- **Q: What is the difference between Spill (Memory) and Spill (Disk)?**
  - *A*: Spill (Memory) is the deserialized uncompressed size in RAM; Spill (Disk) is the compressed serialized size written to local disk.


### More Interview Questions

- **Q: Is spill always a problem?**
  - *A*: No. Some spill is expected for large operations; persistent spill that dominates runtime or exhausts disk is the problem.
- **Q: How can projection reduce spill?**
  - *A*: Fewer columns mean fewer bytes per row, so the sort/aggregation buffer can hold more rows before spilling.

### What to Study Next
Open 06_broadcast_variables.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')